### Better formulations CFLP

A better formulation rather than the standard integer programming formulation for the FLP (with disaggregated constraints) is as follows:

$$
\begin{aligned}
\text{min} \quad & \sum_{i \in I} f_i y_i + \sum_{i \in I} \sum_{j \in J} c_{ij} x_{ij} \\
\text{subject to} \quad & \sum_{i \in I} x_{ij} = 1 \quad \forall j \in J \\
& \sum_{j \in J} d_j x_{ij} \leq q_i \quad \forall i \in I \\
& x_{ij} \leq y_i \quad \forall i \in I,\; \forall j \in J \\
& y_i \in \{0,1\} \quad \forall i \in I \\
& x_{ij} \in \{0,1\} \quad \forall i \in I,\; \forall j \in J
\end{aligned}
$$



In [ ]:
%pip install -q amplpy numpy matplotlib pandas networkx folium
from amplpy import AMPL, ampl_notebook
import numpy as np

# HiGHS is the default. Gurobi requires an AMPL-compatible license.
SOLVER = "highs"  # or "gurobi"
LICENSE_UUID = "default"  # Colab Community Edition; use your UUID locally
runtime = ampl_notebook(modules=[SOLVER], license_uuid=LICENSE_UUID)

def new_ampl():
    return AMPL()

def solve_checked(model):
    model.solve(solver=SOLVER)
    if model.solve_result != "solved":
        raise RuntimeError(f"No proven optimal solution: {model.solve_result}. "
                           "Inspect the solver log before extracting values.")

def values(model, name):
    # Numeric dictionaries keep plotting independent of the solver API.
    return model.var[name].get_values().to_dict()




In [ ]:
# Number of facilities
facilities = 5

# Number of customers
customers = 5

# Sets
I = range(facilities)
J = range(customers)

# Symmetric shipping cost matrix
c = np.array([[0, 3, 3, 6, 3],
              [3, 0, 4, 5, 5],
              [3, 4, 0, 2, 4],
              [6, 5, 2, 0, 5],
              [3, 5, 4, 5, 0]])

# Facility opening cost
f = np.array([25.0, 25.0, 25.0, 25.0, 25.0])

# Demand
d = np.array([10, 8, 5, 8, 12])

# Capacity
q = np.array([15, 15, 15, 15, 15])

In [ ]:
m = new_ampl()
m.eval(r"""
set I;
set J;
param c {I,J};
param d {J} >= 0;
var x {I,J} >= 0;
var y {I} binary;
subject to Assignment {j in J}: sum {i in I} x[i,j] = 1;
param f {I};
minimize Total_Cost: sum {i in I} f[i]*y[i] +
    sum {i in I,j in J} c[i,j]*d[j]*x[i,j];
param q {I} >= 0;
subject to Capacity {i in I}: sum {j in J} d[j]*x[i,j] <= q[i];
subject to Link {i in I,j in J}: x[i,j] <= y[i];
""")
m.set["I"] = list(I)
m.set["J"] = list(J)
m.param["c"] = {(i,j): float(c[i,j]) for i in I for j in J}
m.param["d"] = {j: float(d[j]) for j in J}
m.param["f"] = {i: float(f[i]) for i in I}
m.param["q"] = {i: float(q[i]) for i in I}

solve_checked(m)
x = values(m, "x")
y = values(m, "y")


In [ ]:
if m.solve_result == "solved":
    print("Optimal solution")
    print("Total cost:", m.obj["Total_Cost"].value())
    for i in I:
        if y[i] > 0.1:
            print("Open facility:", i)
else:
    print("Solver status:", m.solve_result)
